# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Follows `checklist.md`.

**Split design.** `data.csv` has no date/time column and every `device_serial` appears exactly once (checked below), so there is neither a temporal ordering nor repeated entities. A stratified random split is therefore the appropriate structure for both the test hold-out and the cross-validation. `device_serial` is an identifier and is dropped. All remaining columns (channel, age, price, daily hours at registration) are known at sale/registration, i.e. at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
SEED=0
df=pd.read_csv('data.csv')
print(df.shape, 'unique serials:', df.device_serial.nunique(), 'missing:', df.isna().sum().sum())
print(df.claim_within_year.mean())
df.describe()

## 1. Hold out the test set first (20%, stratified), then never touch it until the end

In [ ]:
y=df.pop('claim_within_year'); ids=df.pop('device_serial')   # identifier dropped
X=df
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(len(X_train),len(X_test),y_train.mean(),y_test.mean())

## 2. Candidate models (named before any scoring)
1. `baseline` – predicts the training prevalence (trivial).
2. `logreg` – logistic regression, one-hot channel + standardized numerics; grid over C in {0.01, 0.1, 1, 10}.
3. `hgb` – histogram gradient boosting; grid over learning_rate {0.03, 0.1} x max_depth {2, 3} (early-stopping off, 150 iters).

Selection metric: CV average precision (PR-AUC), since the positive class is ~12.6%. Decision threshold for the chosen model is picked on out-of-fold CV predictions by maximizing F1. All preprocessing lives inside the pipelines, so it is fit within each fold.

In [ ]:
num=['device_age_years','price_eur','daily_hours']; cat=['channel']
pre=lambda: ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)])
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
cands={
 'baseline':(Pipeline([('m',DummyClassifier(strategy='prior'))]),{}),
 'logreg':(Pipeline([('p',pre()),('m',LogisticRegression(max_iter=2000))]),{'m__C':[0.01,0.1,1,10]}),
 'hgb':(Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_iter=150,early_stopping=False,random_state=SEED))]),
        {'m__learning_rate':[0.03,0.1],'m__max_depth':[2,3]}),
}
res={}
for n,(pipe,grid) in cands.items():
    gs=GridSearchCV(pipe,grid or {'m__strategy':['prior']},scoring={'ap':'average_precision','auc':'roc_auc'},refit='ap',cv=cv)
    gs.fit(X_train,y_train); res[n]=gs
    print(n,gs.best_params_,'CV AP=%.3f AUC=%.3f'%(gs.best_score_,gs.cv_results_['mean_test_auc'][gs.best_index_]))

## 3. Choice on CV only, including threshold (out-of-fold predictions on training rows)

In [ ]:
best=max(res,key=lambda n:res[n].best_score_)
print('chosen:',best,res[best].best_params_)
oof=cross_val_predict(res[best].best_estimator_,X_train,y_train,cv=StratifiedKFold(5,shuffle=True,random_state=1),method='predict_proba')[:,1]
ths=np.linspace(0.02,0.8,157)
f1s=[f1_score(y_train,oof>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('threshold %.3f, OOF F1 %.3f'%(thr,max(f1s)))

**Decision recorded before test scoring:** the model named above with its CV-selected hyperparameters, features = channel, device_age_years, price_eur, daily_hours, and the F1-maximizing OOF threshold printed above.

## 4. Refit on all training rows; score the test set once

In [ ]:
final=res[best].best_estimator_.fit(X_train,y_train)
p=final.predict_proba(X_test)[:,1]; pred=p>=thr
print('TEST ROC-AUC %.3f  PR-AUC %.3f (prevalence %.3f)'%(roc_auc_score(y_test,p),average_precision_score(y_test,p),y_test.mean()))
print('at threshold %.3f: precision %.3f recall %.3f F1 %.3f'%(thr,precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred)))

## Conclusion
See the printed test metrics above (the headline). Compare the PR-AUC with the prevalence and the baseline's CV score to judge whether the features carry real signal. Caveats: no time column, so we could not test generalization to later sales; `daily_hours` has implausible values (>24 h) left untouched; a single hold-out of 1,400 rows gives a noisy estimate.